# 07 — ML Classification

Classification of grid cells. Mode controlled by `include_other` in `grid.json`:
- **`false`** (default) — Binary: Commercial vs Residential
- **`true`** — 3-class: Commercial vs Residential vs Other

Trains Logistic Regression, XGBoost, and Random Forest on 11 features.
Exports predictions CSV for heatmap visualization.

**Input:** `csv/combined_grid_*.csv`, `grid.json`

**Output:** plots to `outputs/latest/`, predictions to `csv/07_predictions.csv`

In [ ]:
# ── Parameters (injected by papermill) ────────────────
CSV_PATH   = ""   # leave empty to auto-detect
PLOTS_DIR  = "outputs/latest" 

In [ ]:
import matplotlib
matplotlib.use("Agg")

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os, glob

sns.set(color_codes=True, rc={"figure.figsize": (10, 8)})
os.makedirs(PLOTS_DIR, exist_ok=True)

In [ ]:
# ── Load data + class mapping (toggle via grid.json) ──
import json

with open("grid.json", encoding="utf-8") as f:
    _config = json.load(f)
INCLUDE_OTHER = _config.get("include_other", False)
CSV_DIR = _config.get("csv_dir", "csv")

if not CSV_PATH:
    _csvs = sorted(glob.glob(f"{CSV_DIR}/combined_grid_*.csv"))
    if _csvs:
        CSV_PATH = _csvs[-1]
        print(f"Auto-detected: {CSV_PATH}")
    else:
        raise FileNotFoundError(f"No combined_grid CSV found in {CSV_DIR}/")
elif not os.path.exists(CSV_PATH):
    _csvs = sorted(glob.glob(os.path.join(os.path.dirname(CSV_PATH), "combined_grid_*.csv")))
    if _csvs:
        CSV_PATH = _csvs[-1]
        print(f"Fallback: {CSV_PATH}")
    else:
        raise FileNotFoundError(f"CSV not found: {CSV_PATH}")

df = pd.read_csv(CSV_PATH, dtype={"cell_id": str})
print(f"Loaded {len(df)} rows x {df.shape[1]} cols from {CSV_PATH}")

# Class mapping — controlled by include_other toggle
_CLASS_MAP = {
    "Commercial": "Commercial",
    "Mixed-Use": "Residential",
    "Residential": "Residential",
}
if INCLUDE_OTHER:
    _CLASS_MAP.update({
        "Institutional": "Other",
        "Open Space": "Other",
        "Industrial": "Other",
        "Infrastructure": "Other",
    })
    print("Mode: 3-class (Commercial, Residential, Other)")
else:
    print("Mode: Binary (Commercial, Residential) — Other dropped")

df["label"] = df["zone_type"].map(_CLASS_MAP)
df = df[df["label"].notna()].reset_index(drop=True)
print(f"\nClass distribution:")
print(df["label"].value_counts().to_string())

In [ ]:
# ── EDA: class distribution ────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
sns.countplot(data=df, x="label", ax=ax)
ax.set_title("Zone Type Distribution")
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/01_countplot_zone_type.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── EDA: key feature boxplots ─────────────────────────
key_features = ["amenity_density", "shop_density_km2", "tourism_density",
                "landuse_entropy", "avg_floors", "brand_ratio"]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("Key Feature Distributions by Zone Type", fontsize=14)
for ax, feat in zip(axes.flatten(), key_features):
    if feat in df.columns:
        sns.boxplot(data=df, x="label", y=feat, ax=ax)
        ax.set_title(feat)
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/02_feature_boxplots.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── EDA: correlation heatmap ──────────────────────────
FEATURE_COLS = ["amenity_density", "shop_density_km2", "shop_type_entropy",
                "brand_ratio", "tourism_density", "landuse_entropy",
                "amenity_ratio_food_drink", "avg_floors", "avg_yearbuilt",
                "building_count", "total_bldg_area"]

df_feat = df[FEATURE_COLS].copy()
fig, ax = plt.subplots(figsize=(12, 10))
corr = df_feat.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, ax=ax, vmin=-1, vmax=1)
ax.set_title("Feature Correlation Heatmap")
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/03_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Prepare features + labels ─────────────────────────
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Features — only the 11 proven columns
X = df[FEATURE_COLS].copy()
X = X.fillna(X.median())

encoder = LabelEncoder()
y = encoder.fit_transform(df["label"])
class_names = list(encoder.classes_)
print(f"Classes: {class_names}")
print(f"Features: {list(X.columns)}")
print(f"X shape: {X.shape}")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# ── Train/test split ──────────────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train: {X_train.shape[0]}  Test: {X_test.shape[0]}")
print(f"Train class balance: {np.bincount(y_train)}")
print(f"Test class balance:  {np.bincount(y_test)}")

In [ ]:
# ── Logistic Regression ───────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from mlxtend.plotting import plot_confusion_matrix

lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
scores = cross_val_score(lr, X_train, y_train, cv=5)
print(f"LR Cross-val: {scores.mean():.3f} (+/- {scores.std():.3f})")

lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
print(f"Test accuracy: {lr.score(X_test, y_test):.3f}")
print(classification_report(y_test, y_pred_lr, target_names=class_names))

confmatrix = confusion_matrix(y_test, y_pred_lr)
fig, ax = plot_confusion_matrix(conf_mat=confmatrix, colorbar=True,
                                show_absolute=True, show_normed=True,
                                class_names=class_names)
plt.title("Logistic Regression — Confusion Matrix")
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/04_confusion_matrix_lr.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── XGBoost ──────────────────────────────────────────
import xgboost as xgb

xgb_model = xgb.XGBClassifier(random_state=14, eval_metric="mlogloss")
scores = cross_val_score(xgb_model, X_train, y_train, cv=5)
print(f"XGB Cross-val: {scores.mean():.3f} (+/- {scores.std():.3f})")

xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)
print(f"Test accuracy: {xgb_model.score(X_test, y_test):.3f}")
print(classification_report(y_test, y_pred_xgb, target_names=class_names))

confmatrix = confusion_matrix(y_test, y_pred_xgb)
fig, ax = plot_confusion_matrix(conf_mat=confmatrix, colorbar=True,
                                show_absolute=True, show_normed=True,
                                class_names=class_names)
plt.title("XGBoost — Confusion Matrix")
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/05_confusion_matrix_xgb.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Random Forest ─────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(class_weight="balanced", random_state=42)
scores = cross_val_score(rf, X_train, y_train, cv=5)
print(f"RF Cross-val: {scores.mean():.3f} (+/- {scores.std():.3f})")

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print(f"Test accuracy: {rf.score(X_test, y_test):.3f}")
print(classification_report(y_test, y_pred_rf, target_names=class_names))

confmatrix = confusion_matrix(y_test, y_pred_rf)
fig, ax = plot_confusion_matrix(conf_mat=confmatrix, colorbar=True,
                                show_absolute=True, show_normed=True,
                                class_names=class_names)
plt.title("Random Forest — Confusion Matrix")
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/06_confusion_matrix_rf.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Feature importance (Random Forest) ────────────────
importances = pd.Series(rf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(10, 6))
importances.plot(kind="barh", ax=ax)
ax.set_title("Random Forest — Feature Importance")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/07_feature_importance_rf.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Export predictions for heatmap ─────────────────────
# Use LR (balanced performance) to predict all cells
df_all = pd.read_csv(CSV_PATH, dtype={"cell_id": str})

df_all["label"] = df_all["zone_type"].map(_CLASS_MAP)
df_pred = df_all[df_all["label"].notna()].reset_index(drop=True)

X_all = df_pred[FEATURE_COLS].fillna(df_pred[FEATURE_COLS].median())
X_all_scaled = scaler.transform(X_all)

# Predict class + probabilities
df_pred["predicted_zone"] = encoder.inverse_transform(lr.predict(X_all_scaled))
proba = lr.predict_proba(X_all_scaled)

# Always export prob_commercial (used by heatmap diverging colormap)
comm_idx = list(encoder.classes_).index("Commercial")
df_pred["prob_commercial"] = proba[:, comm_idx].round(4)

if INCLUDE_OTHER:
    # Also export per-class probabilities + confidence for 3-class mode
    for i, cls in enumerate(encoder.classes_):
        df_pred[f"prob_{cls.lower()}"] = proba[:, i].round(4)
    df_pred["confidence"] = proba.max(axis=1).round(4)
    extra_cols = ["confidence"] + [f"prob_{c.lower()}" for c in encoder.classes_]
else:
    extra_cols = []

predictions = df_pred[["cell_id", "cell_lat", "cell_lon", "zone_type",
                        "predicted_zone", "prob_commercial"] + extra_cols]
pred_path = f"{CSV_DIR}/07_predictions.csv"
predictions.to_csv(pred_path, index=False, encoding="utf-8")
print(f"Saved predictions: {pred_path} ({len(predictions)} cells)")
for cls in encoder.classes_:
    n = (predictions["predicted_zone"] == cls).sum()
    print(f"  Predicted {cls}: {n} ({100*n/len(predictions):.1f}%)")